In [1]:
%load_ext autoreload
%autoreload 2

import itertools
import sys
import pandas as pd
sys.path.insert(0, '../src')

from recsys.data import load_ratings, make_positive, temporal_split, user_items
from recsys.models import ALSModel
from recsys.metrics import evaluate


ratings = load_ratings('../data')
pos = make_positive(ratings)
train, test = temporal_split(pos)

train_core, valid = temporal_split(train)
valid_truth = user_items(valid)
valid_users = list(valid_truth)

grid = {
    'factors': [16, 32, 64],
    'reg': [30, 100, 300],
    'alpha': [10, 20, 40],
}

rows = []
for factors, reg, alpha in itertools.product(grid['factors'], grid['reg'], grid['alpha']):
    model = ALSModel(factors=factors, reg=reg, alpha=alpha).fit(train_core)
    recs = model.recommend(valid_users, k=50)
    m10 = evaluate(recs, valid_truth, 10)
    m50 = evaluate(recs, valid_truth, 50)
    rows.append({'factors': factors, 'reg': reg, 'alpha': alpha,
                 'recall@10': m10['recall@10'], 'ndcg@10': m10['ndcg@10'],
                 'recall@50': m50['recall@50'], 'ndcg@50': m50['ndcg@50']})
    print(rows[-1], flush=True)

results = pd.DataFrame(rows).sort_values('recall@50', ascending=False)
results.head(10) 

{'factors': 16, 'reg': 30, 'alpha': 10, 'recall@10': 0.07934937570002594, 'ndcg@10': 0.08785085420932745, 'recall@50': 0.2833509383515469, 'ndcg@50': 0.1599852070788075}
{'factors': 16, 'reg': 30, 'alpha': 20, 'recall@10': 0.07399006607730384, 'ndcg@10': 0.08245743854883335, 'recall@50': 0.27518218594883687, 'ndcg@50': 0.154292250415121}
{'factors': 16, 'reg': 30, 'alpha': 40, 'recall@10': 0.06857199690668891, 'ndcg@10': 0.07657300672803988, 'recall@50': 0.26174057972685677, 'ndcg@50': 0.1465513447554902}
{'factors': 16, 'reg': 100, 'alpha': 10, 'recall@10': 0.08041068968036533, 'ndcg@10': 0.09090632747233571, 'recall@50': 0.2816809414658471, 'ndcg@50': 0.16203724104606698}
{'factors': 16, 'reg': 100, 'alpha': 20, 'recall@10': 0.07593439783632645, 'ndcg@10': 0.08551914815686071, 'recall@50': 0.2769471586850771, 'ndcg@50': 0.1570500458664755}
{'factors': 16, 'reg': 100, 'alpha': 40, 'recall@10': 0.07028820012364635, 'ndcg@10': 0.07834403680513824, 'recall@50': 0.26688299049283387, 'ndcg

,factors,reg,alpha,recall@10,ndcg@10,recall@50,ndcg@50
21,64,100,10,0.091019,0.095210,0.311683,0.175064
22,64,100,20,0.086967,0.090431,0.307388,0.170462
12,32,100,10,0.086313,0.093372,0.301605,0.170459
9,32,30,10,0.084924,0.089739,0.300476,0.166623
18,64,30,10,0.086667,0.089591,0.299546,0.166585
23,64,100,40,0.084214,0.087317,0.295684,0.163897
13,32,100,20,0.082798,0.088168,0.295160,0.164625
10,32,30,20,0.079968,0.084439,0.292828,0.161233
19,64,30,20,0.085336,0.087375,0.290118,0.161209
14,32,100,40,0.078259,0.082239,0.285958,0.157574


In [2]:
best = results.iloc[0]
print('Best:', dict(best[['factors', 'reg', 'alpha']]))

final = ALSModel(
    factors=int(best['factors']), reg=float(best['reg']), alpha=float(best['alpha'])
    ).fit(train)

truth = user_items(test)
users = list(truth)
recs = final.recommend(users, k=50)
for k in [10, 20, 50]:
    print(evaluate(recs, truth, k))

Best: {'factors': np.float64(64.0), 'reg': np.float64(100.0), 'alpha': np.float64(10.0)}
{'recall@10': 0.07592583915550757, 'ndcg@10': 0.10207782303367445}
{'recall@20': 0.1360849706187366, 'ndcg@20': 0.12036701483805519}
{'recall@50': 0.2687120261193667, 'ndcg@50': 0.16718816299628128}
